# 다봐요 (Dabwayo) — LaMa 워터마크 제거 **서버** (Colab)

Colab을 **디워터마크 서버**로 띄웁니다. dabwayo의 MCP 툴 `remove_watermark`
(또는 `DABWAYO_LAMA_URL`)이 이 서버로 영상을 보내면, **LaMa 딥 인페인팅**으로
워터마크(Gemini/Veo ✦, ModelScope shutterstock 등)를 지우고 **오디오를 보존한**
mp4를 돌려줍니다.

**계약(contract):**
```
GET  /health       -> {ok, model:'lama', gpu, build}
POST /dewatermark  (multipart) file=<mp4>, regions='[[x,y,w,h],...]',
                    pad, feather, dilate   -> video/mp4 (오디오 포함)
```
**사용:** Runtime → (권장) GPU(T4) → **Run all**. 마지막에 나오는 URL을
`export DABWAYO_LAMA_URL=...` 로 두면 MCP/CLI에서 바로 호출됩니다.

## 1. 설치 (torch는 Colab 기본 내장)

In [ ]:
%pip -q install simple-lama-inpainting fastapi "uvicorn[standard]" nest-asyncio \
  imageio imageio-ffmpeg opencv-python-headless python-multipart
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())

## 2. LaMa 모델 로드 (첫 실행 시 가중치 다운로드)

In [ ]:
from simple_lama_inpainting import SimpleLama
lama = SimpleLama()
print('LaMa ready')

## 3. 디워터마크 API 정의 (`GET /health`, `POST /dewatermark`)

In [ ]:
import tempfile, subprocess, json as _json, traceback
import numpy as np, cv2, imageio.v2 as imageio, imageio_ffmpeg
from PIL import Image, ImageFilter
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.responses import FileResponse

FF = imageio_ffmpeg.get_ffmpeg_exe()

def _process(src, regions, pad, feather, dilate):
    rdr = imageio.get_reader(src); meta = rdr.get_meta_data()
    fps = float(meta.get('fps', 24))
    first = rdr.get_data(0); H, W = first.shape[:2]; rdr.close()
    mask = np.zeros((H, W), np.uint8)
    for (x, y, w, h) in regions:
        x, y, w, h = int(x), int(y), int(w), int(h)
        mask[max(0,y):y+h, max(0,x):x+w] = 255
    if dilate > 0:
        mask = cv2.dilate(mask, np.ones((dilate, dilate), np.uint8))
    ys, xs = np.where(mask > 0)
    if len(xs) == 0:
        raise HTTPException(400, 'regions cover no pixels')
    x0, y0 = max(0, xs.min()-pad), max(0, ys.min()-pad)
    x1, y1 = min(W, xs.max()+pad+1), min(H, ys.max()+pad+1)
    roi_mask = mask[y0:y1, x0:x1]
    roi_mask_pil = Image.fromarray(roi_mask, 'L')
    blend = roi_mask.astype(np.float32) / 255.0
    if feather > 0:
        blend = np.asarray(Image.fromarray((blend*255).astype(np.uint8), 'L')
                           .filter(ImageFilter.GaussianBlur(feather)), np.float32)/255.0
    blend = blend[..., None]
    tmp = tempfile.mktemp(suffix='.mp4')
    wr = imageio.get_writer(tmp, fps=fps, codec='libx264', quality=9,
                            macro_block_size=None)
    for frame in imageio.get_reader(src):
        roi = frame[y0:y1, x0:x1]
        res = lama(Image.fromarray(roi, 'RGB'), roi_mask_pil)
        res = np.asarray(res.convert('RGB').resize((roi.shape[1], roi.shape[0])),
                         np.float32)/255.0
        out = frame.astype(np.float32)/255.0
        out[y0:y1, x0:x1, :3] = res*blend + out[y0:y1, x0:x1, :3]*(1-blend)
        wr.append_data((np.clip(out, 0, 1)*255).astype(np.uint8))
    wr.close()
    final = tempfile.mktemp(suffix='.mp4')
    has_audio = subprocess.run([FF, '-i', src], capture_output=True,
                               text=True).stderr.find('Audio:') != -1
    if has_audio:
        subprocess.run([FF,'-y','-v','error','-i',tmp,'-i',src,'-map','0:v',
                        '-map','1:a','-c:v','copy','-c:a','aac','-b:a','128k',
                        '-shortest',final], check=True)
    else:
        subprocess.run([FF,'-y','-v','error','-i',tmp,'-c','copy',final], check=True)
    return final

app = FastAPI()

@app.get('/health')
def health():
    return {'ok': True, 'model': 'lama', 'build': 'lama-v1',
            'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}

@app.post('/dewatermark')
async def dewatermark(file: UploadFile = File(...), regions: str = Form('[]'),
                      pad: int = Form(48), feather: int = Form(4),
                      dilate: int = Form(9)):
    try:
        regs = _json.loads(regions)
        if regs and all(isinstance(v,(int,float)) for v in regs): regs=[regs]
        src = tempfile.mktemp(suffix='.mp4')
        with open(src,'wb') as f: f.write(await file.read())
        out = _process(src, regs, int(pad), int(feather), int(dilate))
        return FileResponse(out, media_type='video/mp4', filename='dewatermarked.mp4')
    except HTTPException:
        raise
    except Exception:
        tb = traceback.format_exc(); print(tb)
        raise HTTPException(500, 'dewatermark failed:\n' + tb[-1500:])
print('API defined')

## 4. 공개 터널 + 실행 (재실행 안전)

In [ ]:
import nest_asyncio, threading, uvicorn, subprocess, re, time, os, urllib.request, stat, socket
nest_asyncio.apply()

# 재실행 안전: 이전 서버/터널 종료 후 포트 해제 대기 (셀만 다시 돌려도 새 코드 반영)
_old = globals().get('_uvicorn_server')
if _old is not None: _old.should_exit = True
_tun = globals().get('_tunnel_proc')
if _tun is not None:
    try: _tun.terminate()
    except Exception: pass
for _ in range(40):
    s = socket.socket()
    try: s.bind(('0.0.0.0', 8000)); s.close(); break
    except OSError: s.close(); time.sleep(0.5)

_uvicorn_server = uvicorn.Server(uvicorn.Config(app, host='0.0.0.0', port=8000,
                                                log_level='warning'))
threading.Thread(target=_uvicorn_server.run, daemon=True).start(); time.sleep(3)

BIN='/usr/local/bin/cloudflared'
if not os.path.exists(BIN):
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', BIN)
    os.chmod(BIN, os.stat(BIN).st_mode | stat.S_IEXEC)
_tunnel_proc = subprocess.Popen([BIN,'tunnel','--url','http://localhost:8000','--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url=None
for line in _tunnel_proc.stdout:
    m=re.search(r'https://[\w.-]+\.trycloudflare\.com', line)
    if m: url=m.group(0); break
print('\n'+'='*60); print('DABWAYO_LAMA_URL =', url); print('='*60)
print(f"export DABWAYO_LAMA_URL='{url}'")
print('Keep this tab open.')

## 5. (선택) 자가 테스트 — 작은 영상으로 확인

In [ ]:
# 업로드한 mp4 한 개로 왕복 테스트 (예: Veo ✦ 우하단)
# from google.colab import files; up=files.upload(); SRC=list(up.keys())[0]
# import requests, json
# r=requests.post(url+'/dewatermark',
#   files={'file': open(SRC,'rb')},
#   data={'regions': json.dumps([[1132,568,60,62]]), 'pad':'48','feather':'4','dilate':'9'},
#   timeout=900)
# open('clean.mp4','wb').write(r.content); print(r.status_code, len(r.content),'bytes')
# from IPython.display import Video; Video('clean.mp4', embed=True)